In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lifelines import WeibullAFTFitter
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestRegressor
import os

def train_lgbm_models(X_train, y_train):
    """Trains LightGBM Quantile Regressors (P50 and P90)."""
    print(" - Training LightGBM...")
    model_p50 = lgb.LGBMRegressor(objective='quantile', alpha=0.5, n_estimators=100, random_state=42)
    model_p90 = lgb.LGBMRegressor(objective='quantile', alpha=0.9, n_estimators=100, random_state=42)

    model_p50.fit(X_train, y_train)
    model_p90.fit(X_train, y_train)
    return model_p50, model_p90

def train_isolation_forest_pipeline(X_train, y_train):
    """
    Trains an Isolation Forest for anomaly detection, paired with a standard Random Forest.
    Because an Isolation Forest doesn't predict "days" directly, we use the RF for the
    median expectation (P50), and use the Isolation Forest to flag weird shipments
    that require a massive P90 buffer.
    """
    print(" - Training Isolation Forest (Anomaly Detection)...")

    # 1. Base P50 Model (Standard Random Forest Regressor)
    print("   -> Fitting Base RF Model (Median Target)...")
    rf_p50 = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    rf_p50.fit(X_train, y_train)

    # 2. Anomaly Detection Model (The Isolation Forest)
    print("   -> Fitting Isolation Forest (Finding Statistical Outliers)...")
    # contamination=0.05 means we assume roughly 5% of historical shipments were highly anomalous
    iso_forest = IsolationForest(contamination=0.05, random_state=42)
    iso_forest.fit(X_train)

    return rf_p50, iso_forest

def train_survival_model(train_df, features, target_col):
    """
    Trains a Weibull Accelerated Failure Time (AFT) Survival Model.
    Survival analysis treats delivery as an 'event' and calculates the probability of delivery over time.
    """
    print(" - Training Survival Analysis (Weibull AFT)...")
    surv_df = train_df[features + [target_col]].copy()

    # In logistics, every delivered package is an "observed event" (E=1)
    surv_df['Event_Observed'] = 1

    aft = WeibullAFTFitter(penalizer=0.01)
    # Fit the survival model predicting the actual transit days
    aft.fit(surv_df, duration_col=target_col, event_col='Event_Observed')
    return aft

def evaluate_all_models(train_data, test_data, features, target_col):
    """Trains all models and appends their predictions to the test dataset."""
    print("Training Multi-Model Pipeline...")
    X_train, y_train = train_data[features], train_data[target_col]
    X_test = test_data[features]

    # 1. LightGBM
    lgbm_p50, lgbm_p90 = train_lgbm_models(X_train, y_train)
    test_data['Pred_LGBM_P50_Days'] = lgbm_p50.predict(X_test)
    test_data['Pred_LGBM_P90_Days'] = lgbm_p90.predict(X_test)

    # 2. Isolation Forest (Anomaly-based Quoting Hybrid)
    rf_p50, iso_forest = train_isolation_forest_pipeline(X_train, y_train)

    # Get base predictions for P50
    test_data['Pred_IsolationForest_P50_Days'] = rf_p50.predict(X_test)

    # Get anomalies (-1 for anomaly, 1 for normal)
    anomalies = iso_forest.predict(X_test)

    # Construct the P90: If anomaly (-1), add 4 days of safety buffer. If normal (1), add 1 day.
    test_data['Pred_IsolationForest_P90_Days'] = np.where(
        anomalies == -1,
        test_data['Pred_IsolationForest_P50_Days'] + 4.0,
        test_data['Pred_IsolationForest_P50_Days'] + 1.0
    )

    # 3. Survival Analysis
    aft_model = train_survival_model(train_data, features, target_col)
    # Survival analysis outputs percentiles representing when the event (delivery) is 50% or 90% likely
    test_data['Pred_Survival_P50_Days'] = aft_model.predict_percentile(X_test, p=0.5)
    test_data['Pred_Survival_P90_Days'] = aft_model.predict_percentile(X_test, p=0.9)

    print("\nTraining Complete! Multi-model predictions generated.")
    return test_data

# --- Execution Block ---
if __name__ == "__main__":
    train_path = os.path.join("..", "Data", "Processed", "train_data.csv")
    test_path = os.path.join("..", "Data", "Processed", "test_data.csv")
    output_path = os.path.join("..", "Data", "Processed", "Phase3_Advanced_Predictions.csv")

    try:
        train_data = pd.read_csv(train_path)
        test_data = pd.read_csv(test_path)

        target_col = 'Calculated_Actual_Days' if 'Calculated_Actual_Days' in train_data.columns else 'Actual_Transit_Days'
        features = ['Pickup_Day_of_Week', 'Pickup_Month', 'Is_Weekend_Pickup', 'orig_cntry_cd', 'dest_cntry_cd', 'dest_pstl_cd', 'Lane_Historical_Avg', 'Postal_Rolling_7D_Delay_Rate']
        if 'Cluster_ID' in train_data.columns: features.append('Cluster_ID')

        results_df = evaluate_all_models(train_data, test_data, features, target_col)
        results_df.to_csv(output_path, index=False)
        print(f"Saved advanced predictions to: {output_path}")

    except FileNotFoundError:
        print("[ERROR] Could not find training data. Run Phase 3 preprocessing first.")

Training Multi-Model Pipeline...
 - Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000035 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 566
[LightGBM] [Info] Number of data points in the train set: 5824, number of used features: 7
[LightGBM] [Info] Start training from score 7.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000039 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 566
[LightGBM] [Info] Number of data points in the train set: 5824, number of used features: 7
[LightGBM] [Info] Start training from score 12.000000
 - Training Isolation Forest (Anomaly Detection)...
   -> Fitting Base RF Model (Median Target)...
   -> Fitting Isolation Forest (Findi

C:\Users\pimen\Documents\AI and ML\Capstone\.venv\Lib\site-packages\lifelines\fitters\__init__.py:2090: ApproximationWarning: The Hessian was not invertible. We will instead approximate it using the pseudo-inverse.

It's advisable to not trust the variances reported, and to be suspicious of the fitted parameters too.

Some ways to possible ways fix this:

0. Are there any lifelines warnings outputted during the `fit`?
1. Does a particularly large variable need to be centered to 0?
2. Inspect your DataFrame: does everything look as expected? Do you need to add/drop a constant (intercept) column?
3. Is there high-collinearity in the dataset? Try using the variance inflation factor (VIF) to find redundant variables.
4. Trying adding a small penalizer (or changing it, if already present). Example: `WeibullAFTFitter(penalizer=0.01).fit(...)`.
5. Are there any extreme outliers? Try modeling them or dropping them to see if it helps convergence.

  warnings.warn(warning_text, exceptions.Approx